# Reranking with BGE-reranker-v2-m3 (Local)

Two-stage retrieval, run locally on your RTX 4050:

1. **Retrieve wide** with BGE-M3 (bi-encoder) — top-K candidates per query, fast.
2. **Rerank narrow** with BGE-reranker-v2-m3 (cross-encoder) — rescore each
   (query, candidate) pair with full cross-attention, pick the new top-1.

The cross-encoder is more accurate per pair than the bi-encoder but expensive,
so it only ever sees K candidates per query (not the whole corpus).

**What this notebook does:**
- Diagnostic: at what value of K is the gold-best answer present in BGE-M3's top-K?
  (Tells you whether reranking has anything to work with before you commit to it.)
- Compute three numbers per subset on validation:
  - BGE-M3 top-1 (your existing baseline)
  - **Oracle top-K** — the best ROUGE-1 achievable within top-K candidates (the ceiling reranking could reach)
  - **Reranked top-1** — what BGE-reranker actually picks
- Save reranked predictions for the test set.

**Hardware**: RTX 4050 (6 GB). BGE-reranker-v2-m3 is ~2.3 GB in fp16. Fine.

**Network**: requires first-time downloads of BGE-M3 (~2.2 GB) and BGE-reranker-v2-m3 (~2.3 GB).

## 1 — Setup

In [1]:
# !pip install -q -U sentence-transformers transformers torch pandas numpy scikit-learn rouge-score sentencepiece

In [2]:
import os, json, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

device: cuda
gpu: NVIDIA GeForce RTX 4050 Laptop GPU (6.4 GB)


## 2 — Load data

In [3]:
DATA_DIR = Path(".")   # edit if needed
train = pd.read_csv(DATA_DIR/"Train.csv")
val   = pd.read_csv(DATA_DIR/"Val.csv")
test  = pd.read_csv(DATA_DIR/"Test.csv")

QCOL, ACOL, GCOL, IDCOL = "input", "output", "subset", "ID"

for df in (train, val, test):
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna("").astype(str).str.strip()
    if ACOL in df.columns:
        df[ACOL] = df[ACOL].fillna("").astype(str).str.strip()
train = train[(train[QCOL]!="")&(train[ACOL]!="")].reset_index(drop=True)
val   = val[(val[QCOL]!="")&(val[ACOL]!="")].reset_index(drop=True)
print(f"train {len(train)}  val {len(val)}  test {len(test)}")
print(val[GCOL].value_counts())

train 29814  val 6686  test 2618
subset
Eng_Uga    1688
Aka_Gha    1114
Eng_Gha    1104
Lug_Uga     846
Eng_Eth     564
Swa_Ken     518
Amh_Eth     462
Eng_Ken     390
Name: count, dtype: int64


## 3 — Competition scorer (whitespace tokenizer, exact)

Matches the leaderboard.

In [4]:
class WhitespaceTokenizer:
    def tokenize(self, text):
        return [] if text is None else str(text).strip().split()

_SCORER = rouge_scorer.RougeScorer(["rouge1","rougeL"],
                                   tokenizer=WhitespaceTokenizer(), use_stemmer=False)

def rouge1_f1_per_row(preds, refs):
    return np.array([_SCORER.score(str(r), str(p))["rouge1"].fmeasure for p, r in zip(preds, refs)])

def compute_rouge(preds, refs):
    r = rouge1_f1_per_row(preds, refs)
    return {"rouge1_f1": float(r.mean()) if len(r) else 0.0}

def rouge_by_subset(preds, refs, subs, label):
    sub = np.array(subs); rows=[]
    for s in np.unique(sub):
        m = sub == s
        rows.append({"subset":s, "n":int(m.sum()),
                     f"{label}_r1": round(float(rouge1_f1_per_row(
                         [p for p,k in zip(preds,m) if k],
                         [x for x,k in zip(refs,m) if k]).mean()), 4)})
    rows.append({"subset":"OVERALL(micro)", "n":len(preds),
                 f"{label}_r1": round(compute_rouge(preds, refs)["rouge1_f1"], 4)})
    return pd.DataFrame(rows)

## 4 — BGE-M3 retrieval (wide top-K)

Per-subset index, `n_neighbors=K`. Returns parallel lists of candidate answers and
their similarities for every val/test row.

In [5]:
from sentence_transformers import SentenceTransformer

class STEncoder:
    def __init__(self, name): self.model = SentenceTransformer(name, device=DEVICE)
    def encode(self, texts):
        return self.model.encode(texts, normalize_embeddings=True,
                                 show_progress_bar=False, batch_size=64,
                                 convert_to_numpy=True)

class TopKEmbeddingRetriever:
    def __init__(self, encoder, k=20):
        self.encoder = encoder; self.k = k; self.models = {}
    def fit(self, df, qcol, acol, gcol):
        for g, grp in df.groupby(gcol):
            emb = self.encoder.encode(grp[qcol].tolist())
            self.models[g] = {
                "nn": NearestNeighbors(n_neighbors=min(self.k, len(grp)),
                                       metric="cosine").fit(emb),
                "ans": np.array(grp[acol].tolist(), dtype=object),
                "q":   np.array(grp[qcol].tolist(), dtype=object),
            }
        return self
    def retrieve_topk(self, df, qcol, gcol):
        cands = [[] for _ in range(len(df))]
        sims  = [[] for _ in range(len(df))]
        pos = {idx: i for i, idx in enumerate(df.index)}
        for g, grp in df.groupby(gcol):
            m = self.models.get(g) or next(iter(self.models.values()))
            emb = self.encoder.encode(grp[qcol].tolist())
            k_eff = min(self.k, len(m["ans"]))
            dist, idx = m["nn"].kneighbors(emb, n_neighbors=k_eff)
            for row_idx, drow, irow in zip(grp.index, dist, idx):
                i = pos[row_idx]
                cands[i] = [str(m["ans"][j]) for j in irow]
                sims[i]  = [float(1.0 - d) for d in drow]
        return cands, sims

K = 20   # how many candidates the cross-encoder sees per query
bi  = STEncoder("BAAI/bge-m3")
retr = TopKEmbeddingRetriever(bi, k=K).fit(train, QCOL, ACOL, GCOL)
val_cands, val_sims = retr.retrieve_topk(val, QCOL, GCOL)
print(f"Retrieved top-{K} for {len(val_cands)} val rows.")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).
W0528 21:56:04.280000 28524 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Retrieved top-20 for 6686 val rows.


## 5 — Diagnostic: is the gold answer reachable in top-K?

This is the question to answer **before** committing to reranking. Reranking
can only reorder within the candidate set — it cannot conjure an answer that
BGE-M3 didn't retrieve.

For each val row, find the candidate with the highest ROUGE-1 against the
reference. That's the **oracle top-K** ceiling — the best a reranker could
possibly achieve. Compare it to BGE-M3 top-1.

- Big oracle-vs-top1 gap → reranking has real headroom. Pursue it.
- Small gap → the right answer is already at rank 1; reranking won't move the needle. Spend effort elsewhere.

In [6]:
def oracle_topk_predictions(cands_per_row, references):
    """For each row, pick the candidate that best matches the reference (oracle)."""
    out = []
    for cands, ref in zip(cands_per_row, references):
        if not cands:
            out.append(""); continue
        scores = [_SCORER.score(str(ref), str(c))["rouge1"].fmeasure for c in cands]
        out.append(cands[int(np.argmax(scores))])
    return out

bi_top1 = [c[0] if c else "" for c in val_cands]
oracle  = oracle_topk_predictions(val_cands, val[ACOL].tolist())

bi_tbl = rouge_by_subset(bi_top1, val[ACOL].tolist(), val[GCOL].tolist(), "bi_top1")
or_tbl = rouge_by_subset(oracle,  val[ACOL].tolist(), val[GCOL].tolist(), f"oracle_top{K}")
diag = bi_tbl.merge(or_tbl, on=["subset","n"])
diag["headroom"] = (diag[f"oracle_top{K}_r1"] - diag["bi_top1_r1"]).round(4)
print(f"Reranking headroom (oracle top-{K} − BGE-M3 top-1) per subset:")
print(diag.to_string(index=False))

Reranking headroom (oracle top-20 − BGE-M3 top-1) per subset:
        subset    n  bi_top1_r1  oracle_top20_r1  headroom
       Aka_Gha 1114      0.2822           0.3778    0.0956
       Amh_Eth  462      0.1629           0.2916    0.1287
       Eng_Eth  564      0.5476           0.7411    0.1935
       Eng_Gha 1104      0.2826           0.3756    0.0930
       Eng_Ken  390      0.7810           0.9130    0.1320
       Eng_Uga 1688      0.7465           0.9241    0.1776
       Lug_Uga  846      0.4271           0.7335    0.3064
       Swa_Ken  518      0.7662           0.8980    0.1318
OVERALL(micro) 6686      0.4986           0.6566    0.1580


**Read the headroom column.** Subsets with large headroom are the ones reranking
can lift. Subsets with near-zero headroom either already have top-1 right (great) or
don't have a good answer in top-K at all (reranking can't help — that's the Ghana
generation problem, not a ranking problem).

## 6 — Rerank with BGE-reranker-v2-m3

Cross-encoder scores (query, candidate) pairs jointly. The standard companion to
BGE-M3 for reranking; multilingual.

In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=DEVICE, max_length=512)

def rerank_topk(cands_per_row, queries, cross_encoder, batch_size=32):
    """Flatten (query, candidate) pairs, score in one batched call, ungroup per row."""
    flat_pairs, row_lens = [], []
    for q, cands in zip(queries, cands_per_row):
        flat_pairs.extend([(q, c) for c in cands])
        row_lens.append(len(cands))
    if not flat_pairs:
        return [""] * len(queries)
    flat_scores = cross_encoder.predict(flat_pairs, batch_size=batch_size,
                                        show_progress_bar=True, convert_to_numpy=True)
    top1, off = [], 0
    for cands, n in zip(cands_per_row, row_lens):
        if n == 0:
            top1.append(""); continue
        row_scores = flat_scores[off:off+n]; off += n
        top1.append(cands[int(np.argmax(row_scores))])
    return top1

# Watch the progress bar — this scores K * len(val) pairs, batched on GPU.
val_reranked = rerank_topk(val_cands, val[QCOL].tolist(), reranker, batch_size=32)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Batches:   0%|          | 0/4179 [00:00<?, ?it/s]

## 7 — Compare: bi-encoder top-1 vs reranked top-1 vs oracle top-K

In [8]:
rr_tbl = rouge_by_subset(val_reranked, val[ACOL].tolist(), val[GCOL].tolist(), "rerank_top1")
report = diag.merge(rr_tbl, on=["subset","n"])
report = report[["subset","n","bi_top1_r1","rerank_top1_r1",f"oracle_top{K}_r1","headroom"]]
report["gain_from_rerank"] = (report["rerank_top1_r1"] - report["bi_top1_r1"]).round(4)
report["headroom_captured"] = np.where(report["headroom"] > 0,
    ((report["rerank_top1_r1"] - report["bi_top1_r1"]) / report["headroom"]).round(2),
    np.nan)
print(report.to_string(index=False))
print("\nReading: `gain_from_rerank` = what reranking added; `headroom_captured` = "
      "fraction of the available oracle headroom the reranker actually achieved (1.0 = perfect, 0 = none).")

        subset    n  bi_top1_r1  rerank_top1_r1  oracle_top20_r1  headroom  gain_from_rerank  headroom_captured
       Aka_Gha 1114      0.2822          0.3031           0.3778    0.0956            0.0209               0.22
       Amh_Eth  462      0.1629          0.1479           0.2916    0.1287           -0.0150              -0.12
       Eng_Eth  564      0.5476          0.4038           0.7411    0.1935           -0.1438              -0.74
       Eng_Gha 1104      0.2826          0.2848           0.3756    0.0930            0.0022               0.02
       Eng_Ken  390      0.7810          0.6433           0.9130    0.1320           -0.1377              -1.04
       Eng_Uga 1688      0.7465          0.6192           0.9241    0.1776           -0.1273              -0.72
       Lug_Uga  846      0.4271          0.4213           0.7335    0.3064           -0.0058              -0.02
       Swa_Ken  518      0.7662          0.6015           0.8980    0.1318           -0.1647            

## 8 — Inference on test, save submission

Same pipeline applied to test set, then writes a submission CSV with the reranked
top-1 in all three target columns (matching starter format).

In [9]:
test_cands, test_sims = retr.retrieve_topk(test, QCOL, GCOL)
test_reranked = rerank_topk(test_cands, test[QCOL].tolist(), reranker, batch_size=32)

import re
clean = [re.sub(r"<extra_id_\d+>", "", str(p)).strip() for p in test_reranked]
sub = pd.DataFrame({"ID": test[IDCOL], "TargetRLF1": clean, "TargetR1F1": clean, "TargetLLM": clean})
sub.to_csv("submission_bge_rerank.csv", index=False, encoding="utf-8")
print(f"Saved submission_bge_rerank.csv  ({len(sub)} rows)")

Batches:   0%|          | 0/1637 [00:00<?, ?it/s]

Saved submission_bge_rerank.csv  (2618 rows)


In [16]:
# Pick one row and trace its rerank end-to-end
i = 300
q = val[QCOL].iloc[i]
cands = val_cands[i]
ref = val[ACOL].iloc[i]
chosen = val_reranked[i]

# Score candidates ourselves, see what the cross-encoder really thinks
pairs = [(q, c) for c in cands]
scores = reranker.predict(pairs, show_progress_bar=False)

print(f"Query: {q[:120]}")
print(f"Reference: {ref[:120]}")
print(f"Bi-encoder top-1 (chose this): {cands[0][:100]}")
print(f"Reranker top-1 (chose this):   {chosen[:100]}")
print(f"\nReranker scores for the K candidates (higher should = more relevant):")
for rank, (c, s) in enumerate(zip(cands, scores)):
    marker = " <-- reranker chose" if c == chosen else ""
    print(f"  {rank:2d}  score={s:+.3f}  {c[:80]}{marker}")
print(f"\nargmax of scores: index {int(np.argmax(scores))} -> {cands[int(np.argmax(scores))][:80]}")
print(f"argmin of scores: index {int(np.argmin(scores))} -> {cands[int(np.argmin(scores))][:80]}")

Query: So nnipa akuo pɔtee bi wɔ hɔ a wohyia nsɛnnennen soronko wɔ STI ahorow ano sie ho? (mmabun, nna ho adwumayɛfo, nnipa a w
Reference: Akuo binom hyia nsɛnnennen soronko wɔ STI ahorow ano sie ho: Nna mu Adwumayɛfo: Tebea horow a asiane kɛse wom a wɔde wɔn
Bi-encoder top-1 (chose this): Mmeae pii wɔ hɔ a wobɛtumi ayɛ nhwehwɛmu wɔ STI ho wɔ Ghana: Ayaresabea: Aban ayaresabea ne ayaresab
Reranker top-1 (chose this):   Nneɛma bi wɔ hɔ a ɛbɛboa nkurɔfo ma wɔatumi agyina STI a wɔahu no ano: Mmoa Akuw: Sɛ wɔne afoforo a 

Reranker scores for the K candidates (higher should = more relevant):
   0  score=+0.062  Mmeae pii wɔ hɔ a wobɛtumi ayɛ nhwehwɛmu wɔ STI ho wɔ Ghana: Ayaresabea: Aban ay
   1  score=+0.158  Condoms (Kapue): Sɛ wode "condom" bɛdi dwuma berɛ biara a wobɛnya nna (fa awoɔ b
   2  score=+0.001  Titiriw ne nna (ɛtwɛɛ, ɛto, anaa ano). Nneɛma te sɛ nsuo-wɔ-adeɛ (needles) a yɛk
   3  score=+0.479  Yiw, mmabun betumi anya nsɛm a ɛfa sɛnea wobesiw nna mu yareɛ (STI) ano ho bere 
 

In [17]:
# Check: do the candidates fed to the reranker actually belong to the query they were paired with?
import numpy as np

flat_pairs = []
row_lens = []
for q, cs in zip(val[QCOL].tolist(), val_cands):
    flat_pairs.extend([(q, c) for c in cs])
    row_lens.append(len(cs))

# For three sample rows, verify the slice we'd cut out matches the original candidates
offsets = np.cumsum([0] + row_lens)
for i in [0, 100, 500]:
    start, end = offsets[i], offsets[i+1]
    sliced_pairs = flat_pairs[start:end]
    sliced_queries = [p[0] for p in sliced_pairs]
    sliced_cands   = [p[1] for p in sliced_pairs]

    same_query = all(q == val[QCOL].iloc[i] for q in sliced_queries)
    same_cands = sliced_cands == val_cands[i]
    print(f"Row {i}: slice {start}:{end} has {end-start} pairs | "
          f"all paired with same query? {same_query} | "
          f"candidates match val_cands[{i}]? {same_cands}")

Row 0: slice 0:20 has 20 pairs | all paired with same query? True | candidates match val_cands[0]? True
Row 100: slice 2000:2020 has 20 pairs | all paired with same query? True | candidates match val_cands[100]? True
Row 500: slice 10000:10020 has 20 pairs | all paired with same query? True | candidates match val_cands[500]? True


In [18]:
from rouge_score import rouge_scorer
sc = rouge_scorer.RougeScorer(["rouge1"], tokenizer=WhitespaceTokenizer(), use_stemmer=False)

bi_overlap, rr_overlap = [], []
for i in range(len(val)):
    ref = val[ACOL].iloc[i]
    bi_overlap.append(sc.score(ref, val_cands[i][0])["rouge1"].fmeasure)
    rr_overlap.append(sc.score(ref, val_reranked[i])["rouge1"].fmeasure)
print(f"Mean ROUGE-1 of bi-encoder top-1 vs ref: {np.mean(bi_overlap):.4f}")
print(f"Mean ROUGE-1 of reranker top-1 vs ref:   {np.mean(rr_overlap):.4f}")

Mean ROUGE-1 of bi-encoder top-1 vs ref: 0.4986
Mean ROUGE-1 of reranker top-1 vs ref:   0.4356


In [ ]:
# find rows where bi-encoder >> reranker
deltas = [b - r for b, r in zip(bi_overlap, rr_overlap)]
worst = np.argsort(deltas)[-5:][::-1]
for i in worst:
    print(f"\n--- row {i} | subset {val[GCOL].iloc[i]} | bi={bi_overlap[i]:.3f} rr={rr_overlap[i]:.3f} ---")
    print(f"Q:   {val[QCOL].iloc[i][:120]}")
    print(f"REF: {val[ACOL].iloc[i][:200]}")
    print(f"BI:  {val_cands[i][0][:200]}")
    print(f"RR:  {val_reranked[i][:200]}")


--- row 1616 | subset Eng_Eth | bi=1.000 rr=0.000 ---
Q:   How do boys and girls avoid problems of can men carry chlamydia without knowing?
REF: This is a question about, Chlamydia. Chlamydia is curable with antibiotics. Testing with urine or swabs is needed, and prevention includes condoms and regular health checks.
BI:  This is a question about, Chlamydia. Chlamydia is curable with antibiotics. Testing with urine or swabs is needed, and prevention includes condoms and regular health checks.
RR:  Yes. Men can carry chlamydia without symptoms (asymptomatic infection).

--- row 5764 | subset Lug_Uga | bi=1.000 rr=0.000 ---
Q:   Ngeri ki Abantu ze basobola okukolaganamu n'abo abalina Obuwuka obuleeta Obulwadde bw'Ekikaba nga tebabasosodde?
REF: Okukolagana n'abalwadde b'Endwadde z'Ekikaba awatali kubasosola kyetaagisa nnyo okutumbula okusaasira, okutegeera, na buli muntu okwenyigiramu. Eno y'emu ku nnambika ku ngeri y'okuzimba enkolagana eya
BI:  Okukolagana n'abalwadde b'Endwadde z'Eki

: 

## 9 — Where this fits in the larger plan

- **If `headroom_captured` is high overall (≥0.6):** reranking is doing its job. Stack
  it as the default retriever and route Ghana subsets to your mT0/RAG-reader from
  the other notebook.
- **If `headroom_captured` is low:** the cross-encoder isn't picking well on this
  data. Either the answers are too similar for it to distinguish (try a larger model
  or a domain-specific reranker), or this is the wrong lever for your hard subsets.
- **If `headroom` itself is near zero for a subset:** reranking can't help there.
  The right answer isn't in top-K. This is the generation-needed signal — that
  subset wants the RAG reader, not a better ranker.

This notebook tells you both *whether* reranking is worth it (section 5) and *how
much* it actually buys (section 7) before you commit it to your submission stack.